# Eksperimen Machine Learning - Wine Quality Dataset

**Nama:** gaji  
**Dataset:** Wine Quality (Red Wine)  
**Objective:** Membangun model klasifikasi biner untuk memprediksi kualitas wine (good/bad)  

## Tahapan Eksperimen:
1. Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Feature Engineering
5. Model Building (baseline)

---

## 1. Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

print('Libraries imported successfully!')

In [ ]:
# Load dataset
df = pd.read_csv('../wine_quality_raw/winequality-red.csv', sep=';')

print(f'Dataset Shape: {df.shape}')
print(f'\nKolom: {list(df.columns)}')
print(f'\nTipe Data:\n{df.dtypes}')
df.head(10)

In [ ]:
# Info dataset
df.info()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Statistik deskriptif
df.describe().round(3)

In [ ]:
# Cek missing values
print('Missing Values per Kolom:')
print(df.isnull().sum())
print(f'\nTotal Missing Values: {df.isnull().sum().sum()}')

In [ ]:
# Cek duplikat
print(f'Jumlah duplikat: {df.duplicated().sum()}')

In [ ]:
# Distribusi target (quality)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
df['quality'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Distribusi Wine Quality', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
df['quality'].value_counts().sort_index().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporsi Wine Quality', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Distribusi fitur
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.2f}')
    axes[i].legend(fontsize=8)

plt.suptitle('Distribusi Seluruh Fitur', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, square=True, linewidths=0.5)
plt.title('Correlation Heatmap - Wine Quality Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot untuk deteksi outlier
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    sns.boxplot(data=df, y=col, ax=axes[i], color='lightcoral')
    axes[i].set_title(col, fontsize=11, fontweight='bold')

plt.suptitle('Boxplot - Deteksi Outlier', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Korelasi fitur terhadap quality
plt.figure(figsize=(10, 6))
quality_corr = df.corr()['quality'].drop('quality').sort_values()
quality_corr.plot(kind='barh', color=['red' if x < 0 else 'green' for x in quality_corr])
plt.title('Korelasi Fitur terhadap Quality', fontsize=14, fontweight='bold')
plt.xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Copy dataframe
df_processed = df.copy()
print(f'Shape awal: {df_processed.shape}')

In [ ]:
# Handle missing values (jika ada)
missing = df_processed.isnull().sum().sum()
print(f'Missing values: {missing}')

if missing > 0:
    numeric_cols = df_processed.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        df_processed[col].fillna(df_processed[col].median(), inplace=True)
    print(f'Missing values setelah treatment: {df_processed.isnull().sum().sum()}')
else:
    print('Tidak ada missing values.')

In [ ]:
# Remove duplicates
print(f'Duplikat sebelum: {df_processed.duplicated().sum()}')
df_processed = df_processed.drop_duplicates().reset_index(drop=True)
print(f'Shape setelah hapus duplikat: {df_processed.shape}')

In [ ]:
# Handle outliers menggunakan IQR clipping
feature_cols = [col for col in df_processed.columns if col != 'quality']

for col in feature_cols:
    Q1 = df_processed[col].quantile(0.25)
    Q3 = df_processed[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = ((df_processed[col] < lower) | (df_processed[col] > upper)).sum()
    if outliers > 0:
        df_processed[col] = df_processed[col].clip(lower=lower, upper=upper)
        print(f'{col}: {outliers} outlier(s) clipped to [{lower:.2f}, {upper:.2f}]')

In [ ]:
# Buat target biner: quality >= 7 = good (1), else bad (0)
df_processed['quality_label'] = (df_processed['quality'] >= 7).astype(int)

print(f'Distribusi target biner:')
print(df_processed['quality_label'].value_counts())
print(f'\nPersentase good wine: {df_processed["quality_label"].mean()*100:.1f}%')

# Drop kolom quality asli
df_processed = df_processed.drop('quality', axis=1)

In [ ]:
# Train-test split
X = df_processed.drop('quality_label', axis=1)
y = df_processed['quality_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_train distribution:\n{y_train.value_counts()}')
print(f'y_test distribution:\n{y_test.value_counts()}')

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), 
    columns=X_train.columns, 
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), 
    columns=X_test.columns, 
    index=X_test.index
)

print('Feature scaling selesai!')
X_train_scaled.describe().round(3)

In [ ]:
# Save preprocessed data
import os
output_dir = 'wine_quality_preprocessing'
os.makedirs(output_dir, exist_ok=True)

X_train_scaled.to_csv(os.path.join(output_dir, 'X_train.csv'), index=False)
X_test_scaled.to_csv(os.path.join(output_dir, 'X_test.csv'), index=False)
y_train.to_csv(os.path.join(output_dir, 'y_train.csv'), index=False)
y_test.to_csv(os.path.join(output_dir, 'y_test.csv'), index=False)

print(f'Data preprocessed disimpan di: {output_dir}/')
print('Files:', os.listdir(output_dir))

## 4. Baseline Model

In [ ]:
# Baseline model - Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

y_pred = rf_model.predict(X_test_scaled)

print('=== Baseline Model: Random Forest ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Bad', 'Good']))

In [ ]:
# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Bad', 'Good'], yticklabels=['Bad', 'Good'])
plt.title('Confusion Matrix - Baseline Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
feature_imp = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feature_imp.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Feature Importance - Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Kesimpulan

- Dataset Wine Quality memiliki 1599 sampel dengan 11 fitur numerik
- Tidak ditemukan missing values, terdapat beberapa duplikat yang sudah dihapus
- Outlier ditangani menggunakan IQR clipping
- Target dikonversi menjadi klasifikasi biner (good: quality >= 7, bad: quality < 7)
- Baseline Random Forest menunjukkan performa yang baik
- Langkah selanjutnya: hyperparameter tuning dan model tracking dengan MLflow